[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabriceyhc/emerging-substances-la/blob/main/notebooks/emergence_validation.ipynb)

# Emergence validation: annotated alarm review

Reviews every substance that at least one of five detection methods has ever alarmed on, annotated against the same raw death-count line.

**Data sources** (regenerate with the commands shown if stale):

| file | command | grain |
|---|---|---|
| `results/trends/rising_substances.csv` | `emerging trends rank` | current EB05++TS score per substance, used only to order Part 1's dropdown and Part 2's grid |
| `results/trends/emergence_by_quarter.csv` | `emerging trends emergence-table` | substance x quarter, raw death counts |
| `results/trends/emergence_by_year.csv` | `emerging trends emergence-table` | substance x year, raw death counts (the requested deliverable table; not plotted here, quarterly is used instead so alarm markers land on the right x-position) |
| `results/benchmark/method_timeline.csv` | `emerging benchmark method-timeline` | substance x as-of-quarter x method, score + alarm flag |

**The six methods**, an escalating ladder of sophistication (`docs/GPS_V2_DESIGN.md`), not six arbitrary picks:

| label | what it is | id in `emerging/validation/benchmark.py` |
|---|---|---|
| `n/E` | raw ratio, no shrinkage at all -- plotted for context but **excluded** from "has this substance ever fired" below: with no shrinkage at all it alarms on nearly every new arrival (872 of the sweep's ~3,700 substance-quarters), so using it as a filter would defeat the point of picking substances worth reviewing | `ratio` |
| `EB05` | classic single-gate EB05, raw counts | `eb05` |
| `EB05+` | dual-gated EB05 (share up **and** own-count up), raw counts | `eb05-dual` |
| `TreeScan` | substance-leaf recurrence interval, solo | `treescan` |
| `NB-Trend` | log-linear Poisson trend, Wald z of the fitted slope | `nb-trend` |
| `EB05++TS` | **the deployed detector** -- weighted + role-discounted + spatial-fused dual-gated EB05, TreeScan-vetoed at RI 10 | not a single benchmark.py id; read from `trends alarms`'s own cache (`eb05_sweep.csv`), the one reading here expensive enough that `alarms` already pays that cost |

Every method's raw score is on an incomparable scale to the others (a ratio, a posterior percentile, a recurrence interval, a Wald z) -- this notebook never plots them against each other, only *when each one alarmed*, as markers on the one line that is comparable: the substance's own raw death count.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import Dropdown, Output
from IPython.display import clear_output, display

%matplotlib inline

# Locally (VS Code, JupyterLab) this notebook is assumed to stay at
# notebooks/emergence_validation.ipynb, one level under the repo root, and
# nothing below runs. Colab starts with an empty runtime with no checkout of
# this repo at all, so clone it there -- the three CSVs this notebook reads
# are aggregate counts already committed to the repo (`emerging/paths.py`'s
# rule: no case numbers, no coordinates in `results/`), so cloning is safe.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    REPO = Path("/content/emerging-substances-la")
    if REPO.exists() and not (REPO / ".git").exists():
        # A previous run's clone was interrupted (e.g. it hung and the cell
        # was stopped) and left a partial, non-git directory behind -- retry
        # from scratch rather than silently reusing a broken checkout.
        !rm -rf {REPO}
    if not REPO.exists():
        # Fail fast instead of hanging: with no credential helper configured,
        # a clone that unexpectedly needs auth would otherwise sit forever on
        # a username prompt Colab's `!` shell can never answer.
        os.environ["GIT_TERMINAL_PROMPT"] = "0"
        !git clone -q https://github.com/fabriceyhc/emerging-substances-la.git {REPO}
    assert (REPO / ".git").exists(), "clone failed -- see the git output above"
    ROOT = REPO
else:
    ROOT = Path("..")
RESULTS = ROOT / "results"

In [ ]:
timeline = pd.read_csv(RESULTS / "benchmark" / "method_timeline.csv", parse_dates=["as_of"])
quarterly = pd.read_csv(RESULTS / "trends" / "emergence_by_quarter.csv", index_col=0)
quarterly.columns = pd.to_datetime(quarterly.columns)
ranking = pd.read_csv(RESULTS / "trends" / "rising_substances.csv", index_col=0)["eb05"]

# "Fired" = alarmed under at least one of the four non-ensemble methods or
# the deployed detector -- n/E excluded (see the table above). This is a
# strict superset of `alarm_history.csv`'s substances (which is EB05++TS
# alone): TreeScan and NB-Trend in particular each flag substances the
# deployed detector's own dual gate + veto never do, and part of this
# review's point is to see where the methods disagree, not just where the
# deployed one already agrees with itself.
ALARM_METHODS = ["EB05", "EB05+", "TreeScan", "NB-Trend", "EB05++TS"]
ever_alarmed = timeline.loc[timeline["method"].isin(ALARM_METHODS) & timeline["alarm"],
                           "substance"].unique()

# Ordered by current EB05++TS score, descending -- same convention as Part
# 2's grid and `geo export`'s one-hot columns, so all three read substances
# in the same priority order. A few names here alarmed only under TreeScan
# or NB-Trend and never scored highly on EB05++TS itself, so they sit near
# the bottom of this list despite being exactly the disagreements worth
# reviewing -- that ordering choice, not a bug.
FIRED = (ranking.reindex(ever_alarmed)
         .sort_values(ascending=False, na_position="last").index.tolist())
print(f"{len(FIRED)} substances have ever fired one of {ALARM_METHODS}:")
print(FIRED)

## Part 1 -- annotated review: substances that fired

One panel per substance, picked from the dropdown -- any substance at least one of EB05, EB05+, TreeScan, NB-Trend or EB05++TS has ever alarmed on, not only the deployed detector's own list. The black line is the raw quarterly death count (`emergence_by_quarter.csv`); each colored row of markers above it is one method, placed at the as-of quarters where *that method's own* alarm condition was true (`method_timeline.csv`'s `alarm` column) -- not at a shared y-value tied to the line, so a quiet quarter's markers don't collide with the line itself. `n/E` still plots as context (grey dots) even though it doesn't gate which substances appear here.

In [ ]:
METHOD_STYLE = {
    "n/E":      dict(marker="o", color="#b8b7b2"),
    "EB05":     dict(marker="s", color="#2a78d6"),
    "EB05+":    dict(marker="^", color="#1baf7a"),
    "TreeScan": dict(marker="D", color="#eda100"),
    "NB-Trend": dict(marker="v", color="#7a4fd1"),
    "EB05++TS": dict(marker="*", color="#eb6834"),
}
METHOD_ORDER = list(METHOD_STYLE)


def plot_substance(substance):
    series = quarterly.loc[substance]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(series.index, series.values, color="#0b0b0b", linewidth=1.6, zorder=2)
    ax.fill_between(series.index, series.values, color="#0b0b0b", alpha=0.05, zorder=1)

    sub = timeline[timeline["substance"] == substance]
    ymax = max(series.max(), 1)
    for i, method in enumerate(METHOD_ORDER):
        fired_q = sub[(sub["method"] == method) & sub["alarm"]]
        if not len(fired_q):
            continue
        style = METHOD_STYLE[method]
        y = ymax * (1.06 + 0.05 * i)
        ax.scatter(fired_q["as_of"], [y] * len(fired_q), label=method,
                   marker=style["marker"], color=style["color"],
                   edgecolors="white", linewidths=0.6, s=45, zorder=3)

    ax.set_title(f"{substance} -- quarterly deaths, six methods' alarm quarters",
                loc="left", fontsize=12)
    ax.set_ylim(0, ymax * 1.4)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, ncol=6, loc="upper left",
             bbox_to_anchor=(0, -0.08), fontsize=8.5)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# Wired by hand (Dropdown.observe -> Output) instead of ipywidgets.interact():
# interact's own auto-generated Output area is known to be flaky specifically
# on Colab -- the widget renders, but selecting a new value doesn't reliably
# re-run the callback's matplotlib output back into view, so every substance
# past the default rendered as a blank re-selection. Driving the Output
# widget explicitly avoids relying on that internal wiring, and works the
# same way locally (VS Code, JupyterLab) as on Colab.
_out = Output()


def _render(change=None):
    with _out:
        clear_output(wait=True)
        plot_substance(_dropdown.value)


_dropdown = Dropdown(options=FIRED, description="substance")
_dropdown.observe(_render, names="value")
display(_dropdown, _out)
_render()

## Part 2 -- appendix: all 211 substances, unannotated

Completeness/audit view, not a validation read: the ~176 substances not in Part 1's dropdown never alarmed under any of the five methods and are mostly flat or sparse lines with nothing to disagree about. No alarm markers here -- just the raw quarterly count, so a reviewer can confirm nothing in the long tail was overlooked, without 211 panels each carrying a legend.

Ordered by *current* EB05++TS score, descending -- same convention as Part 1's dropdown, `geo export`'s one-hot columns, and `emergence_by_year.csv`'s rows.

In [ ]:
substances = quarterly.index.tolist()  # already current-EB05++TS-ordered
ncols = 14
nrows = -(-len(substances) // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.15, nrows * 0.75))
for ax, name in zip(axes.flat, substances):
    s = quarterly.loc[name]
    ax.plot(s.index, s.values, color="#2a78d6", linewidth=0.9)
    ax.fill_between(s.index, s.values, color="#2a78d6", alpha=0.12)
    ax.set_title(name, fontsize=5.5, pad=1.5)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
for ax in axes.flat[len(substances):]:
    ax.axis("off")

fig.suptitle("All 211 substances, raw quarterly deaths -- ordered by current "
            "EB05++TS score (audit view, no method annotation)",
            fontsize=10, y=1.005)
fig.tight_layout()
plt.show()